### Part 1
![saved_map.png](saved_map.png)

### Part 2
a) No, the maximal reactions here are different for all maximal reaction activities. This does not change the fact of mass balance constraint - the fluxes are still balanced. Numbers on the map indicate how much flux this reaction is able to carry out. They are the upper bounds for the model.

b) The values are "nd" (no data) or 0.00. The 0.00 represents the enzyme constraint put in the model, which does not allow the reaction to carry any flux. nd means that the measurement was not possible.

In [4]:
import pandas as pd
import cobra

model = cobra.io.load_json_model("e_coli_core-1.json")

df = pd.read_csv("KEN3170_Assignment_2026_e_coli_core_expression.csv")

In [6]:
# implementing maximum reaction activity for all reactions in model

for reaction_id, value in zip(df["# Reaction ID"], df[" reaction activity [mmol/gDW/h] "]):
    reaction = model.reactions.get_by_id(reaction_id)

    if value == "nd":
        continue

    if reaction.id == "ATPM":
        reaction.upper_bound = value
        continue

    if reaction.reversibility:
        model.reactions.get_by_id(reaction_id).bounds = (-value, value)
    elif not reaction.reversibility:
        model.reactions.get_by_id(reaction_id).bounds = (0, value)

model.reactions.EX_glc__D_e.lower_bound = -1000

for reaction in model.reactions:
    print(reaction.id, reaction.bounds)

PFK (0, 13.1)
PFL (0, 0.0)
PGI (-11.1, 11.1)
PGK (-24.0, 24.0)
PGL (0, 7.3)
ACALD (0, 0.0)
AKGt2r (0, 0.0)
PGM (-21.7, 21.7)
PIt2r (-5.2, 5.2)
ALCD2x (0, 0.0)
ACALDt (-1000.0, 1000.0)
ACKr (-2.5, 2.5)
PPC (0, 3.6)
ACONTa (-21.4, 21.4)
ACONTb (-21.4, 21.4)
ATPM (8.39, 1000.0)
PPCK (0, 13.3)
ACt2r (-3.6, 3.6)
PPS (0, 3.1)
ADK1 (-27.4, 27.4)
AKGDH (0, 26.7)
ATPS4r (-80.1, 80.1)
PTAr (-4.47, 4.47)
PYK (0, 28.2)
BIOMASS_Ecoli_core_w_GAM (0.0, 1000.0)
PYRt2 (0, 0.0)
CO2t (-1000.0, 1000.0)
RPE (-6.3, 6.3)
CS (0, 21.4)
RPI (-5.6, 5.6)
SUCCt2_2 (0, 0.0)
CYTBD (0, 41.1)
D_LACt2 (0, 0.0)
ENO (-29.3, 29.3)
SUCCt3 (0, 0.0)
ETOHt2r (-1000.0, 1000.0)
SUCDi (0, 27.3)
SUCOAS (-19.4, 19.4)
TALA (-4.5, 4.5)
THD2 (0, 6.5)
TKT1 (-3.5, 3.5)
TKT2 (-3.5, 3.5)
TPI (-70.0, 70.0)
EX_ac_e (0.0, 1000.0)
EX_acald_e (0.0, 1000.0)
EX_akg_e (0.0, 1000.0)
EX_co2_e (-1000.0, 1000.0)
EX_etoh_e (0.0, 1000.0)
EX_for_e (0.0, 1000.0)
EX_fru_e (0.0, 1000.0)
EX_fum_e (0.0, 1000.0)
EX_glc__D_e (-1000, 1000.0)
EX_gln__L_e (0.0, 

In [ ]:
##  safety check
print(model.reactions.EX_glc__D_e.bounds)
print(model.reactions.ATPM.bounds)

(-1000, 1000.0)
(8.39, 1000.0)


### Part 3


In [ ]:
## 3a

solution = model.optimize()

print("Maximum biomass production rate:", solution.objective_value)

Maximum biomass production rate: 0.8732862458582367


Max biomass production rate is 0.8733 h⁻¹ (4 d.p.)

In [ ]:
## 3b

model.reactions.EX_glc__D_e.lower_bound =-5

print("Glucose exchange bounds:", model.reactions.EX_glc__D_e.bounds)

Glucose exchange bounds: (-5, 1000.0)


The -5 represents the limit of glucose the cell can get from its environment, measured in mmol/gDW/h. In exercise 2, the expression-based constraints were tied to the maximum capacity of internal reactions, which is known through enzyme levels. In this case, a hard limit of 5 mmol/gDW/h limits glucose uptake, which in extension limits energy and bioimass production.

In [10]:
##  3c
solution_limited_glucose = model.optimize()

print("Maximum biomass production rate with glucose limit:", solution_limited_glucose.objective_value)

Maximum biomass production rate with glucose limit: 0.41559777509290663


With this hard limit, maximal biomass production decreased from 0.8733 to 0.4156mmol/gDW/h , which is  over 52% of a decrease, showing that limited glucose uptake can harm or slow growth. This makes sense because, as mentioned in 3b, with less glucose entering the cell, there is less energy available for biomass production, and therefore the maximum growth rate will be lower.